# Skeleton visualization across input sources

Visualize one frame from each skeleton source in NTU-25 layout, then again after `scale_normalize_to_ntu`. Use this to eyeball that conversions and normalization are sane before training.

Sources:
- **MotionBert** (`motionBert_cropped_iou`, 17 joints) → NTU-25
- **MediaPipe World** (`world_mp_cropped_iou`, 33 joints) → NTU-25
- **MediaPipe Camera** (`camera_mp_cropped_iou`, 33 joints) → NTU-25  *(broken for pretrained encoders; shown for comparison)*
- **NTU120** reference sample from `MAMP/ntu120/NTU120_XSub.npz`

In [ ]:
import os
import sys

# Add project root so `from data.skeletonMapping import ...` works
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers '3d' projection)

from data.skeletonMapping import (
    convertVideoMBtoNTU,
    convertVideoMPtoNTU,
    scale_normalize_to_ntu,
    NTU_REF_SPINE_LEN,
)
from data.PoseDataset import load_video_h5

print('Project root:', PROJECT_ROOT)
print('NTU_REF_SPINE_LEN:', NTU_REF_SPINE_LEN)

## NTU-25 skeleton edges

Joint index reference:
- 0: hipMid, 1: spineMid, 2: neck, 3: nose (head), 20: shoulderMid
- 4–7: left arm (shoulder, elbow, wrist, hand), 21–22: left hand tips
- 8–11: right arm, 23–24: right hand tips
- 12–15: left leg (hip, knee, ankle, toe), 16–19: right leg

In [ ]:
NTU_EDGES = [
    # spine
    (0, 1), (1, 20), (20, 2), (2, 3),
    # left arm + hand tips
    (20, 4), (4, 5), (5, 6), (6, 7), (7, 21), (7, 22),
    # right arm + hand tips
    (20, 8), (8, 9), (9, 10), (10, 11), (11, 23), (11, 24),
    # left leg
    (0, 12), (12, 13), (13, 14), (14, 15),
    # right leg
    (0, 16), (16, 17), (17, 18), (18, 19),
]


def to_numpy(x):
    if hasattr(x, 'cpu'):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def plot_skeleton(ax, joints25, title='', color='C0', equal_lim=0.8):
    """
    joints25: (25, 3) NTU-layout single frame.
    Plots joints + bones in 3D with consistent axes.
    """
    j = to_numpy(joints25)
    x, y, z = j[:, 0], j[:, 1], j[:, 2]

    # Bones
    for a, b in NTU_EDGES:
        ax.plot([x[a], x[b]], [z[a], z[b]], [y[a], y[b]], color=color, lw=1.5)
    # Joints
    ax.scatter(x, z, y, color=color, s=15)
    # Joint 0 = hipMid (highlight) and joint 3 = head (highlight)
    ax.scatter([x[0]], [z[0]], [y[0]], color='black', s=40, label='hipMid (0)')
    ax.scatter([x[3]], [z[3]], [y[3]], color='red', s=40, label='nose (3)')

    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('z')
    ax.set_zlabel('y (up)')
    # Center on hip, use shared limit for visual comparability
    cx, cy, cz = x[0], y[0], z[0]
    ax.set_xlim(cx - equal_lim, cx + equal_lim)
    ax.set_ylim(cz - equal_lim, cz + equal_lim)
    ax.set_zlim(cy - equal_lim, cy + equal_lim)
    ax.legend(loc='upper right', fontsize=7)

## Load one frame from each source

In [ ]:
# --- Adjust these paths to your environment ---
H5_LIST = '/code/jjiang23/pathml/aim2_balance/processed_files/all_h5_files.txt'
NTU120_NPZ = '/code/jjiang23/BalanceTestThesis/models/encoders/MAMP/ntu120/NTU120_XSub.npz'
FRAME_IDX = 0  # which frame within the chosen video
FILE_IDX  = 1  # which h5 file from the list

with open(H5_LIST, 'r') as f:
    h5_files = [line.strip() for line in f.readlines()]
h5_path = h5_files[FILE_IDX]
print('Loading from:', h5_path)

# MotionBert -> NTU (17 -> 25)
mb_kps     = load_video_h5(h5_path, 'motionBert_cropped_iou',  allPhases=False)[0]
mb_ntu     = convertVideoMBtoNTU(mb_kps)

# MediaPipe world -> NTU (33 -> 25)
mpw_kps    = load_video_h5(h5_path, 'world_mp_cropped_iou',   allPhases=False)[0]
mpw_ntu    = convertVideoMPtoNTU(mpw_kps)

# MediaPipe camera -> NTU (33 -> 25)  -- broken for pretrained encoders, shown for context
mpc_kps    = load_video_h5(h5_path, 'camera_mp_cropped_iou',  allPhases=False)[0]
mpc_ntu    = convertVideoMPtoNTU(mpc_kps)

# NTU120 sample (already 25 joints)
npz = np.load(NTU120_NPZ)
x_train = npz['x_train']                         # (N, T=300, 150)
ntu_sample = x_train[0].reshape(300, 2, 25, 3)   # (T, M, V, C)
ntu120_frame = ntu_sample[FRAME_IDX, 0]          # body 0, frame 0  -> (25, 3)

print('Shapes (T,25,3):')
for name, k in [('mb_ntu', mb_ntu), ('mpw_ntu', mpw_ntu), ('mpc_ntu', mpc_ntu)]:
    print(f'  {name:8s}: {tuple(k.shape)}')
print(f'  ntu120  : (25, 3) one frame')

## Raw conversions (before encoder scale-normalization)

Same frame from each source, plotted at the same axis range. You should see roughly human-shaped skeletons with head up (red), hip at center (black). Camera_mp will look distorted because z is in different units than x,y.

In [ ]:
fig = plt.figure(figsize=(20, 5))

samples = [
    ('NTU120 reference',         ntu120_frame,            'C2', 1.0),
    ('MB -> NTU (raw)',          mb_ntu[FRAME_IDX],       'C0', 1.0),
    ('world_mp -> NTU (raw)',    mpw_ntu[FRAME_IDX],      'C1', 1.0),
    ('camera_mp -> NTU (raw)',   mpc_ntu[FRAME_IDX],      'C3', 1.5),
]
for i, (title, frame, color, lim) in enumerate(samples):
    ax = fig.add_subplot(1, 4, i + 1, projection='3d')
    plot_skeleton(ax, frame, title=title, color=color, equal_lim=lim)

plt.tight_layout()
plt.show()

## After `scale_normalize_to_ntu`

Same frame, after bringing spine length to NTU120 mean (~0.196 m). MB and world_mp should now look proportional to the NTU120 reference. Camera_mp's anisotropic z still looks wrong — single-scalar scaling can't fix per-axis unit mismatch (this is why we hard-error on it for MAMP/MAE).

In [ ]:
def normalize_one_frame(frame_25x3):
    """Apply scale_normalize_to_ntu to a single frame by adding a batch+time dim."""
    arr = to_numpy(frame_25x3)
    b = arr[None, None]                          # (1, 1, 25, 3)
    out = scale_normalize_to_ntu(b)              # numpy branch
    return out[0, 0]

fig = plt.figure(figsize=(20, 5))
samples_norm = [
    ('NTU120 reference',                ntu120_frame,                          'C2', 0.8),
    ('MB -> NTU (scale-normed)',        normalize_one_frame(mb_ntu[FRAME_IDX]), 'C0', 0.8),
    ('world_mp -> NTU (scale-normed)',  normalize_one_frame(mpw_ntu[FRAME_IDX]),'C1', 0.8),
    ('camera_mp -> NTU (scale-normed)', normalize_one_frame(mpc_ntu[FRAME_IDX]),'C3', 1.0),
]
for i, (title, frame, color, lim) in enumerate(samples_norm):
    ax = fig.add_subplot(1, 4, i + 1, projection='3d')
    plot_skeleton(ax, frame, title=title, color=color, equal_lim=lim)

plt.tight_layout()
plt.show()

## Quick sanity checks (numbers)

Spine length and y-axis convention should match across MB / world_mp / NTU120 after normalization.

In [ ]:
def spine_len(frame):
    f = to_numpy(frame)
    return float(np.linalg.norm(f[20] - f[1]))

def axis_check(frame, name):
    f = to_numpy(frame)
    print(f'{name:35s}  spine={spine_len(f):.4f}   '
          f'head_y={f[3,1]:+.3f}  hip_y={f[0,1]:+.3f}  toe_y={f[15,1]:+.3f}')

print('--- Raw (after y-flip in convert*, before scale-norm) ---')
axis_check(mb_ntu[FRAME_IDX],   'MB raw')
axis_check(mpw_ntu[FRAME_IDX],  'world_mp raw')
axis_check(mpc_ntu[FRAME_IDX],  'camera_mp raw (broken)')
axis_check(ntu120_frame,        'NTU120 reference')

print('\n--- After scale_normalize_to_ntu ---')
axis_check(normalize_one_frame(mb_ntu[FRAME_IDX]),  'MB normed')
axis_check(normalize_one_frame(mpw_ntu[FRAME_IDX]), 'world_mp normed')
axis_check(normalize_one_frame(mpc_ntu[FRAME_IDX]), 'camera_mp normed (still broken)')

print(f'\nTarget spine length (NTU120 mean): {NTU_REF_SPINE_LEN}')

## Optional: multi-frame overlay

Plot every Nth frame from a single source overlaid in one 3D view. Good for spotting whether the conversion is temporally stable (no joint flipping/swapping).

In [ ]:
def overlay_frames(seq_T25x3, step=20, color='C0', title=''):
    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    seq = to_numpy(seq_T25x3)
    T = seq.shape[0]
    for t in range(0, T, step):
        alpha = 0.2 + 0.8 * (t / max(T - 1, 1))
        for a, b in NTU_EDGES:
            ax.plot([seq[t, a, 0], seq[t, b, 0]],
                    [seq[t, a, 2], seq[t, b, 2]],
                    [seq[t, a, 1], seq[t, b, 1]],
                    color=color, lw=1.0, alpha=alpha)
    ax.set_title(title)
    ax.set_xlabel('x'); ax.set_ylabel('z'); ax.set_zlabel('y (up)')
    plt.tight_layout()
    plt.show()

overlay_frames(mb_ntu,  step=30, color='C0', title='MB -> NTU (every 30th frame)')
overlay_frames(mpw_ntu, step=30, color='C1', title='world_mp -> NTU (every 30th frame)')